# Confidently Wrong: Calibration and Silent Failure of Lightweight Chest X-ray Classifiers under Physics-Informed Acquisition Degradation, with a Conformal Quality Gate

**Target journal:** IJEEEMI. **Dataset:** PneumoniaMNIST at 224×224 (derived from the Kermany pediatric chest X-ray dataset; official train/val/test splits).

## How to run
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. Run the first time with `QUICK_TEST = True` (about 5–10 minutes) to check everything works.
3. Then set `QUICK_TEST = False` and choose **Runtime → Run all** (about 1.5–2.5 hours).
4. If Colab disconnects, simply **Run all** again: finished training runs and evaluation conditions are skipped automatically (everything is saved to your Google Drive).
5. At the end, download **`IJEEEMI_CXR_results.zip`** from the `IJEEEMI_CXR` folder in your Google Drive and upload it to the chat.

Do not share passwords; Colab only asks you to approve Drive access in a pop-up.

In [ ]:
# ===== 1. Setup & configuration =====
QUICK_TEST = False  # <-- full experiment (set True only for a quick check)

import os, sys, json, time, subprocess
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)

ROOT = "/content/drive/MyDrive/IJEEEMI_CXR" + ("_quick" if QUICK_TEST else "")
CFG = dict(
    ROOT=ROOT,
    DATA_DIR="/content/drive/MyDrive/IJEEEMI_CXR_data",
    IMG_SIZE=224,
    MODELS=["mobilenet_v3_large", "efficientnet_b0", "resnet18"],
    SEEDS=[0] if QUICK_TEST else [0, 1, 2],
    EPOCHS=1 if QUICK_TEST else 15,
    PATIENCE=5,
    BATCH=64, LR=3e-4, WD=1e-4,
    CONF_THR=0.9,             # 'confident' threshold for confident errors
    K_TTA=2 if QUICK_TEST else 8,
    ALPHAS_CONF=[0.005, 0.01, 0.02],   # target rates for accepted CONFIDENT errors
    ALPHAS_ERR=[0.05, 0.10, 0.15],     # target rates for accepted errors of any kind
    N_TRIALS=20 if QUICK_TEST else 200,
    TRAIN_SUBSET=512 if QUICK_TEST else None,
    TEST_SUBSET=128 if QUICK_TEST else None,
    DOSE_SUBSET=64 if QUICK_TEST else 128,
    DOSE_R=4 if QUICK_TEST else 16,
)
for sub in ["ckpt", "preds", "tables", "figures", "logs"]:
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
json.dump(CFG, open(os.path.join(ROOT, "logs", "config.json"), "w"), indent=2)
print(json.dumps(CFG, indent=2))

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "ROOT": "/content/drive/MyDrive/IJEEEMI_CXR",
  "DATA_DIR": "/content/drive/MyDrive/IJEEEMI_CXR_data",
  "IMG_SIZE": 224,
  "MODELS": [
    "mobilenet_v3_large",
    "efficientnet_b0",
    "resnet18"
  ],
  "SEEDS": [
    0,
    1,
    2
  ],
  "EPOCHS": 15,
  "PATIENCE": 5,
  "BATCH": 64,
  "LR": 0.0003,
  "WD": 0.0001,
  "CONF_THR": 0.9,
  "K_TTA": 8,
  "ALPHAS_CONF": [
    0.005,
    0.01,
    0.02
  ],
  "ALPHAS_ERR": [
    0.05,
    0.1,
    0.15
  ],
  "N_TRIALS": 200,
  "TRAIN_SUBSET": null,
  "TEST_SUBSET": null,
  "DOSE_SUBSET": 128,
  "DOSE_R": 16
}


In [ ]:
%%writefile core.py
# ===== Physics-informed degradations, metrics, and conformal risk control =====
# All degradations act on display images v in [0,1] (bright = high attenuation),
# which we map to a line-integral proxy p = v * P_MAX (Beer-Lambert: I = I0 * exp(-p)).
import numpy as np
from scipy.ndimage import gaussian_filter, convolve1d, zoom
from sklearn.metrics import roc_auc_score

P_MAX = 4.0        # max line integral represented by v = 1
N0 = 1.0e4         # photons per pixel (unattenuated) at nominal (full) dose
SIGMA_E = 5.0      # electronic (Gaussian) noise, in photon-count units

DEGRADATIONS = {
    # name: (list of 5 severity parameters, human-readable parameter name)
    "dose":     ([0.5, 0.25, 0.1, 0.05, 0.02], "dose fraction d"),
    "scatter":  ([0.25, 0.5, 1.0, 2.0, 4.0],   "scatter-to-primary ratio"),
    "contrast": ([0.85, 0.7, 0.55, 0.4, 0.25], "subject-contrast factor kappa"),
    "overexp":  ([1.5, 2.0, 3.0, 5.0, 8.0],    "detector gain error e"),
    "motion":   ([3, 5, 9, 13, 19],            "motion length (px)"),
}

def _to_p(v):   return v.astype(np.float32) * P_MAX
def _to_v(p):   return np.clip(p / P_MAX, 0.0, 1.0).astype(np.float32)

def deg_dose(v, d, rng):
    """Quantum (Poisson) + electronic (Gaussian) noise at dose fraction d."""
    p = _to_p(v)
    lam = d * N0 * np.exp(-p)
    n = rng.poisson(lam).astype(np.float32) + rng.normal(0, SIGMA_E, size=lam.shape).astype(np.float32)
    n = np.maximum(n, 1.0)
    return _to_v(-np.log(n / (d * N0)))

def deg_scatter(v, spr, rng=None):
    """Additive low-frequency scatter S with mean(S)/mean(I_p) = SPR (fixed display window)."""
    p = _to_p(v)
    ip = np.exp(-p)                                   # (B,H,W), I0 = 1
    B, H, W = ip.shape
    f = 8                                             # blur on a downsampled grid (fast, equivalent for a wide kernel)
    small = zoom(ip, (1, 1 / f, 1 / f), order=1)
    small = gaussian_filter(small, sigma=(0, H / (8 * f), W / (8 * f)), mode="reflect")
    s = zoom(small, (1, H / small.shape[1], W / small.shape[2]), order=1)
    s = s * (spr * ip.mean(axis=(1, 2), keepdims=True) / np.maximum(s.mean(axis=(1, 2), keepdims=True), 1e-8))
    return _to_v(-np.log(ip + s))

def deg_contrast(v, kappa, rng=None):
    """kVp-like subject-contrast reduction: p' = mean(p) + kappa (p - mean(p))."""
    p = _to_p(v)
    m = p.mean(axis=(1, 2), keepdims=True)
    return _to_v(m + kappa * (p - m))

def deg_overexp(v, e, rng=None):
    """Detector gain error e > 1 with saturation at I_sat = 1, reconstructed with nominal calibration."""
    ip = np.exp(-_to_p(v))
    idet = np.minimum(e * ip, 1.0)
    return _to_v(-np.log(np.maximum(idet, 1e-8)))

def deg_motion(v, L, rng=None):
    """Vertical (respiratory-like) linear motion blur of length L pixels."""
    k = np.ones(int(L), dtype=np.float32) / float(L)
    return convolve1d(v.astype(np.float32), k, axis=1, mode="reflect")

DEG_FN = {"dose": deg_dose, "scatter": deg_scatter, "contrast": deg_contrast,
          "overexp": deg_overexp, "motion": deg_motion}

def conditions():
    out = [("clean", 0, None)]
    for name, (params, _) in DEGRADATIONS.items():
        for i, prm in enumerate(params):
            out.append((name, i + 1, prm))
    return out

def apply_condition(v, name, prm, seed):
    """v: float32 (B,H,W) in [0,1]. Deterministic given seed."""
    if name == "clean":
        return v.astype(np.float32)
    rng = np.random.default_rng(seed)
    return DEG_FN[name](v, prm, rng)

def noise_estimate(v):
    """Immerkaer (1996) fast noise sigma estimate, per image. Model-free quality score."""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    from scipy.ndimage import convolve
    out = np.empty(v.shape[0], dtype=np.float32)
    H, W = v.shape[1:]
    for i in range(v.shape[0]):
        r = convolve(v[i], k, mode="reflect")
        out[i] = np.sqrt(np.pi / 2) * np.abs(r).sum() / (6 * (W - 2) * (H - 2))
    return out

# ---------------- metrics (binary, single logit) ----------------
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def ece_binary(prob_pos, y, n_bins=15):
    pred = (prob_pos >= 0.5).astype(int)
    conf = np.where(pred == 1, prob_pos, 1 - prob_pos)
    correct = (pred == y).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins[1:-1], right=True), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

def metrics_from_logits(logits, y, T=1.0, conf_thr=0.9):
    p = sigmoid(logits / T)
    pred = (p >= 0.5).astype(int)
    conf = np.where(pred == 1, p, 1 - p)
    wrong = pred != y
    eps = 1e-7
    out = {
        "acc": float((~wrong).mean()),
        "bal_acc": float(0.5 * ((pred[y == 1] == 1).mean() + (pred[y == 0] == 0).mean())),
        "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "ece": float(ece_binary(p, y)),
        "brier": float(np.mean((p - y) ** 2)),
        "nll": float(-np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))),
        "conf_err_rate": float((wrong & (conf >= conf_thr)).mean()),
        "frac_err_confident": float((wrong & (conf >= conf_thr)).sum() / max(wrong.sum(), 1)),
        "mean_conf_wrong": float(conf[wrong].mean()) if wrong.any() else np.nan,
        "sens": float((pred[y == 1] == 1).mean()),
        "spec": float((pred[y == 0] == 0).mean()),
    }
    return out

# ---------------- conformal risk control (Angelopoulos et al., 2024) ----------------
def crc_threshold(scores_cal, loss_fn_cal, alpha, B=1.0):
    """
    Units i = 1..n (base images). Gate ACCEPTS a sample when score <= lam.
    loss_fn_cal(lam) -> array (n,) of per-unit losses in [0,B], non-decreasing in lam.
    Returns largest lam with (n * Rhat(lam) + B) / (n + 1) <= alpha  (or -inf: reject all).
    """
    n = loss_fn_cal(np.inf).shape[0]
    grid = np.unique(scores_cal.ravel())
    best = -np.inf
    # risk is monotone in lam -> binary search over grid
    lo, hi = 0, len(grid) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r = (n * loss_fn_cal(grid[mid]).mean() + B) / (n + 1)
        if r <= alpha:
            best = grid[mid]; lo = mid + 1
        else:
            hi = mid - 1
    return best

def unit_loss(scores, conf_err, lam):
    """scores, conf_err: (n_units, n_conditions). Loss = fraction of a unit's versions that are accepted confident errors."""
    return ((scores <= lam) & conf_err).mean(axis=1)

def run_crc(scores, conf_err, alpha, n_trials=200, cal_frac=0.5, seed=0):
    """scores, conf_err: (n_units, n_cond). Random image-level splits. Returns list of dicts."""
    rng = np.random.default_rng(seed)
    n = scores.shape[0]
    res = []
    for t in range(n_trials):
        perm = rng.permutation(n)
        nc = int(cal_frac * n)
        c, te = perm[:nc], perm[nc:]
        lam = crc_threshold(scores[c], lambda l: unit_loss(scores[c], conf_err[c], l), alpha)
        acc_mask = scores[te] <= lam
        res.append({
            "trial": t, "lambda": float(lam),
            "test_risk": float((acc_mask & conf_err[te]).mean()),
            "accept_rate": float(acc_mask.mean()),
            "risk_no_gate": float(conf_err[te].mean()),
            "cond_conf_err_among_accepted": float((acc_mask & conf_err[te]).sum() / max(acc_mask.sum(), 1)),
        })
    return res


Writing core.py


In [ ]:
# ===== 2. Data =====
import numpy as np, importlib, core
importlib.reload(core)
from medmnist import PneumoniaMNIST

def load(split):
    ds = PneumoniaMNIST(split=split, download=True, size=CFG["IMG_SIZE"], root=CFG["DATA_DIR"])
    X = np.asarray(ds.imgs)
    if X.ndim == 4: X = X[..., 0]
    y = np.asarray(ds.labels).reshape(-1).astype(np.int64)
    return X.astype(np.uint8), y

Xtr, ytr = load("train"); Xva, yva = load("val"); Xte, yte = load("test")
if CFG["TRAIN_SUBSET"]:
    idx = np.random.default_rng(0).permutation(len(Xtr))[:CFG["TRAIN_SUBSET"]]; Xtr, ytr = Xtr[idx], ytr[idx]
if CFG["TEST_SUBSET"]:
    idx = np.random.default_rng(0).permutation(len(Xte))[:CFG["TEST_SUBSET"]]; Xte, yte = Xte[idx], yte[idx]
for n, X, y in [("train", Xtr, ytr), ("val", Xva, yva), ("test", Xte, yte)]:
    print(f"{n:5s} {X.shape}  pneumonia={y.mean():.3f}")
np.save(os.path.join(ROOT, "preds", "y_test.npy"), yte)

train (4708, 224, 224)  pneumonia=0.742
val   (524, 224, 224)  pneumonia=0.742
test  (624, 224, 224)  pneumonia=0.625


In [ ]:
# ===== 3. Models, augmentation, prediction helpers =====
import math, torch.nn as nn, torch.nn.functional as F, torchvision.models as tvm
DEV = "cuda"
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)

def build(name):
    if name == "mobilenet_v3_large":
        m = tvm.mobilenet_v3_large(weights="DEFAULT"); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights="DEFAULT"); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "resnet18":
        m = tvm.resnet18(weights="DEFAULT"); m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "densenet121":
        m = tvm.densenet121(weights="DEFAULT"); m.classifier = nn.Linear(m.classifier.in_features, 1)
    else:
        raise ValueError(name)
    return m.to(DEV)

def to_input(x):                       # x: (B,1,H,W) in [0,1]
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

def random_affine(x, rot_deg, trans, scale, gen=None):
    B = x.shape[0]
    r = lambda: torch.rand(B, device=x.device, generator=gen) * 2 - 1
    a = r() * rot_deg * math.pi / 180; s = 1 + r() * scale; tx = r() * trans * 2; ty = r() * trans * 2
    theta = torch.stack([torch.stack([torch.cos(a) / s, -torch.sin(a) / s, tx], 1),
                         torch.stack([torch.sin(a) / s,  torch.cos(a) / s, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="border", align_corners=False)

def train_augment(x):
    x = random_affine(x, 7, 0.05, 0.05)
    B = x.shape[0]
    c = 1 + (torch.rand(B, 1, 1, 1, device=x.device) * 2 - 1) * 0.1
    b = (torch.rand(B, 1, 1, 1, device=x.device) * 2 - 1) * 0.05
    m = x.mean(dim=(2, 3), keepdim=True)
    return ((x - m) * c + m + b).clamp(0, 1)

@torch.no_grad()
def predict_logits(model, V, batch=128, tta_seed=None, amp=True):
    """V: numpy (N,H,W), uint8 or float in [0,1]. Returns float64 logits (N,).
    If tta_seed is given, applies a mild, seeded random affine perturbation (same for every model)."""
    model.eval(); out = []
    gen = None
    if tta_seed is not None:
        gen = torch.Generator(device=DEV); gen.manual_seed(int(tta_seed))
    for i in range(0, len(V), batch):
        xb = torch.from_numpy(np.ascontiguousarray(V[i:i + batch])).to(DEV)
        xb = (xb.float() / 255.0 if xb.dtype == torch.uint8 else xb.float()).unsqueeze(1)
        if gen is not None:
            xb = random_affine(xb, 5, 0.03, 0.03, gen=gen)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lo = model(to_input(xb)).squeeze(1)
        out.append(lo.float().cpu())
    return torch.cat(out).numpy().astype(np.float64)
print("helpers ready")

helpers ready


In [ ]:
# ===== 4. Training (resumable; checkpoints in Drive) =====
import pandas as pd, random
from sklearn.metrics import roc_auc_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr).float()
log_path = os.path.join(ROOT, "logs", "train_log.csv")

for name in CFG["MODELS"]:
    for seed in CFG["SEEDS"]:
        key = f"{name}_s{seed}"
        done_p = os.path.join(ROOT, "ckpt", key + "_done.json")
        best_p = os.path.join(ROOT, "ckpt", key + "_best.pt")
        last_p = os.path.join(ROOT, "ckpt", key + "_last.pt")
        if os.path.exists(done_p):
            print("skip (done):", key); continue
        set_seed(seed)
        model = build(name)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"])
        scaler = torch.amp.GradScaler("cuda")
        start, best_auc, best_ep, bad = 0, -1.0, -1, 0
        if os.path.exists(last_p):
            st = torch.load(last_p, map_location=DEV, weights_only=False)
            model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
            sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
            start, best_auc, best_ep, bad = st["epoch"] + 1, st["best_auc"], st["best_ep"], st["bad"]
            print(f"resume {key} from epoch {start}")
        t0 = time.time()
        g = torch.Generator(); g.manual_seed(seed)
        for ep in range(start, CFG["EPOCHS"]):
            model.train(); perm = torch.randperm(len(Xtr_t), generator=g); tl = 0.0
            for i in range(0, len(perm), CFG["BATCH"]):
                idx = perm[i:i + CFG["BATCH"]]
                xb = Xtr_t[idx].to(DEV, non_blocking=True).float().div_(255).unsqueeze(1)
                yb = ytr_t[idx].to(DEV)
                xb = train_augment(xb)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = F.binary_cross_entropy_with_logits(model(to_input(xb)).squeeze(1).float(), yb)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(idx)
            sched.step()
            vl = predict_logits(model, Xva); vauc = roc_auc_score(yva, vl)
            improved = vauc > best_auc
            if improved:
                best_auc, best_ep, bad = vauc, ep, 0
                torch.save(model.state_dict(), best_p)
            else:
                bad += 1
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                            scaler=scaler.state_dict(), epoch=ep, best_auc=best_auc, best_ep=best_ep, bad=bad), last_p)
            row = dict(key=key, model=name, seed=seed, epoch=ep, train_loss=tl / len(perm), val_auroc=vauc)
            pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
            print(f"{key} ep{ep} loss={tl/len(perm):.4f} valAUROC={vauc:.4f}{' *' if improved else ''}")
            if bad >= CFG["PATIENCE"]:
                print("early stop"); break
        json.dump(dict(key=key, best_val_auroc=best_auc, best_epoch=best_ep,
                       train_minutes=(time.time() - t0) / 60,
                       n_params=sum(p.numel() for p in model.parameters())), open(done_p, "w"), indent=2)
        if os.path.exists(last_p): os.remove(last_p)
        del model, opt; torch.cuda.empty_cache()
print("training finished")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 75.8MB/s]


mobilenet_v3_large_s0 ep0 loss=0.1311 valAUROC=0.9925 *
mobilenet_v3_large_s0 ep1 loss=0.0528 valAUROC=0.9942 *
mobilenet_v3_large_s0 ep2 loss=0.0326 valAUROC=0.9974 *
mobilenet_v3_large_s0 ep3 loss=0.0197 valAUROC=0.9981 *
mobilenet_v3_large_s0 ep4 loss=0.0150 valAUROC=0.9975
mobilenet_v3_large_s0 ep5 loss=0.0184 valAUROC=0.9978
mobilenet_v3_large_s0 ep6 loss=0.0127 valAUROC=0.9984 *
mobilenet_v3_large_s0 ep7 loss=0.0049 valAUROC=0.9991 *
mobilenet_v3_large_s0 ep8 loss=0.0058 valAUROC=0.9985
mobilenet_v3_large_s0 ep9 loss=0.0056 valAUROC=0.9986
mobilenet_v3_large_s0 ep10 loss=0.0026 valAUROC=0.9993 *
mobilenet_v3_large_s0 ep11 loss=0.0014 valAUROC=0.9990
mobilenet_v3_large_s0 ep12 loss=0.0024 valAUROC=0.9991
mobilenet_v3_large_s0 ep13 loss=0.0016 valAUROC=0.9989
mobilenet_v3_large_s0 ep14 loss=0.0018 valAUROC=0.9990
mobilenet_v3_large_s1 ep0 loss=0.1425 valAUROC=0.9947 *
mobilenet_v3_large_s1 ep1 loss=0.0497 valAUROC=0.9964 *
mobilenet_v3_large_s1 ep2 loss=0.0311 valAUROC=0.9980 *
mob

100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


efficientnet_b0_s0 ep0 loss=0.1425 valAUROC=0.9991 *
efficientnet_b0_s0 ep1 loss=0.0692 valAUROC=0.9993 *
efficientnet_b0_s0 ep2 loss=0.0420 valAUROC=0.9996 *
efficientnet_b0_s0 ep3 loss=0.0265 valAUROC=0.9996
efficientnet_b0_s0 ep4 loss=0.0229 valAUROC=0.9997 *
efficientnet_b0_s0 ep5 loss=0.0191 valAUROC=0.9992
efficientnet_b0_s0 ep6 loss=0.0177 valAUROC=0.9995
efficientnet_b0_s0 ep7 loss=0.0132 valAUROC=0.9986
efficientnet_b0_s0 ep8 loss=0.0104 valAUROC=0.9991
efficientnet_b0_s0 ep9 loss=0.0046 valAUROC=0.9989
early stop
efficientnet_b0_s1 ep0 loss=0.1477 valAUROC=0.9988 *
efficientnet_b0_s1 ep1 loss=0.0606 valAUROC=0.9960
efficientnet_b0_s1 ep2 loss=0.0464 valAUROC=0.9998 *
efficientnet_b0_s1 ep3 loss=0.0347 valAUROC=0.9990
efficientnet_b0_s1 ep4 loss=0.0280 valAUROC=0.9993
efficientnet_b0_s1 ep5 loss=0.0218 valAUROC=0.9995
efficientnet_b0_s1 ep6 loss=0.0173 valAUROC=0.9990
efficientnet_b0_s1 ep7 loss=0.0104 valAUROC=0.9991
early stop
efficientnet_b0_s2 ep0 loss=0.1539 valAUROC=0.99

100%|██████████| 44.7M/44.7M [00:00<00:00, 178MB/s]


resnet18_s0 ep0 loss=0.1103 valAUROC=0.9978 *
resnet18_s0 ep1 loss=0.0608 valAUROC=0.9981 *
resnet18_s0 ep2 loss=0.0344 valAUROC=0.9994 *
resnet18_s0 ep3 loss=0.0256 valAUROC=0.9982
resnet18_s0 ep4 loss=0.0195 valAUROC=0.9989
resnet18_s0 ep5 loss=0.0209 valAUROC=0.9988
resnet18_s0 ep6 loss=0.0157 valAUROC=0.9996 *
resnet18_s0 ep7 loss=0.0079 valAUROC=0.9995
resnet18_s0 ep8 loss=0.0040 valAUROC=0.9996
resnet18_s0 ep9 loss=0.0026 valAUROC=0.9997 *
resnet18_s0 ep10 loss=0.0022 valAUROC=0.9996
resnet18_s0 ep11 loss=0.0020 valAUROC=0.9997
resnet18_s0 ep12 loss=0.0029 valAUROC=0.9997
resnet18_s0 ep13 loss=0.0013 valAUROC=0.9997
resnet18_s0 ep14 loss=0.0026 valAUROC=0.9996
early stop
resnet18_s1 ep0 loss=0.1085 valAUROC=0.9983 *
resnet18_s1 ep1 loss=0.0494 valAUROC=0.9966
resnet18_s1 ep2 loss=0.0337 valAUROC=0.9985 *
resnet18_s1 ep3 loss=0.0334 valAUROC=0.9984
resnet18_s1 ep4 loss=0.0232 valAUROC=0.9980
resnet18_s1 ep5 loss=0.0215 valAUROC=0.9987 *
resnet18_s1 ep6 loss=0.0112 valAUROC=0.9990 

In [ ]:
# ===== 5. Load best models + temperature scaling on clean validation set =====
from scipy.special import expit
def fit_temperature(logits, y):
    Ts = np.exp(np.linspace(np.log(0.05), np.log(20), 600))
    nll = [-np.mean(y * np.log(expit(logits / T) + 1e-7) + (1 - y) * np.log(1 - expit(logits / T) + 1e-7)) for T in Ts]
    return float(Ts[int(np.argmin(nll))])

MODELS, TEMPS, info = {}, {}, []
for name in CFG["MODELS"]:
    for seed in CFG["SEEDS"]:
        key = f"{name}_s{seed}"
        m = build(name); m.load_state_dict(torch.load(os.path.join(ROOT, "ckpt", key + "_best.pt"), map_location=DEV)); m.eval()
        MODELS[key] = m
        vl = predict_logits(m, Xva); TEMPS[key] = fit_temperature(vl, yva)
        d = json.load(open(os.path.join(ROOT, "ckpt", key + "_done.json")))
        d["temperature"] = TEMPS[key]; info.append(d)
        print(key, "T =", round(TEMPS[key], 3))
json.dump(TEMPS, open(os.path.join(ROOT, "tables", "temperatures.json"), "w"), indent=2)
pd.DataFrame(info).to_csv(os.path.join(ROOT, "tables", "training_summary.csv"), index=False)

mobilenet_v3_large_s0 T = 1.869
mobilenet_v3_large_s1 T = 2.215
mobilenet_v3_large_s2 T = 1.53
efficientnet_b0_s0 T = 1.036
efficientnet_b0_s1 T = 0.616
efficientnet_b0_s2 T = 0.66
resnet18_s0 T = 1.357
resnet18_s1 T = 1.168
resnet18_s2 T = 1.215


In [ ]:
# ===== 6. Evaluate every model under every degradation (resumable per condition) =====
V_test = Xte.astype(np.float32) / 255.0
CONDS = core.conditions()
rng_ex = np.random.default_rng(1)
ex_idx = [int(np.where(yte == 0)[0][0]), int(np.where(yte == 1)[0][0])]   # one normal, one pneumonia example
for ci, (cname, lvl, prm) in enumerate(CONDS):
    fp = os.path.join(ROOT, "preds", f"cond_{cname}_L{lvl}.npz")
    if os.path.exists(fp):
        continue
    t0 = time.time()
    Vd = core.apply_condition(V_test, cname, prm, seed=1000 + ci)
    save = dict(noise=core.noise_estimate(Vd), examples=Vd[ex_idx].astype(np.float16))
    for key, m in MODELS.items():
        T = TEMPS[key]
        base = predict_logits(m, Vd)
        tta = np.stack([predict_logits(m, Vd, tta_seed=10_000 * ci + k) for k in range(CFG["K_TTA"])])
        p_all = expit(np.concatenate([base[None], tta]) / T)
        save[f"{key}__logit"] = base
        save[f"{key}__tta_std"] = p_all.std(axis=0)
        save[f"{key}__tta_flip"] = ((p_all[1:] >= 0.5) != (p_all[0] >= 0.5)).mean(axis=0)
    np.savez_compressed(fp, **save)
    print(f"[{ci+1}/{len(CONDS)}] {cname} L{lvl} ({prm}) done in {time.time()-t0:.0f}s")
print("evaluation finished")

[1/26] clean L0 (None) done in 31s
[2/26] dose L1 (0.5) done in 35s
[3/26] dose L2 (0.25) done in 34s
[4/26] dose L3 (0.1) done in 34s
[5/26] dose L4 (0.05) done in 34s
[6/26] dose L5 (0.02) done in 34s
[7/26] scatter L1 (0.25) done in 33s
[8/26] scatter L2 (0.5) done in 33s
[9/26] scatter L3 (1.0) done in 33s
[10/26] scatter L4 (2.0) done in 33s
[11/26] scatter L5 (4.0) done in 33s
[12/26] contrast L1 (0.85) done in 31s
[13/26] contrast L2 (0.7) done in 31s
[14/26] contrast L3 (0.55) done in 31s
[15/26] contrast L4 (0.4) done in 31s
[16/26] contrast L5 (0.25) done in 31s
[17/26] overexp L1 (1.5) done in 31s
[18/26] overexp L2 (2.0) done in 31s
[19/26] overexp L3 (3.0) done in 31s
[20/26] overexp L4 (5.0) done in 31s
[21/26] overexp L5 (8.0) done in 31s
[22/26] motion L1 (3) done in 31s
[23/26] motion L2 (5) done in 31s
[24/26] motion L3 (9) done in 31s
[25/26] motion L4 (13) done in 31s
[26/26] motion L5 (19) done in 31s
evaluation finished


In [ ]:
# ===== 7. Dose-scaling experiment (tests the delta-method prediction Var[logit] ~ 1/d) =====
fp = os.path.join(ROOT, "preds", "dose_scaling.npz")
if not os.path.exists(fp):
    sub = np.random.default_rng(7).permutation(len(V_test))[:CFG["DOSE_SUBSET"]]
    Vs = V_test[sub]; doses = np.array([100.0, 30.0, 10.0, 3.0, 1.0, 0.5, 0.25, 0.1, 0.05, 0.02])
    out = dict(doses=doses, sub=sub)
    for key, m in MODELS.items():
        arr = np.zeros((len(doses), CFG["DOSE_R"], len(sub)))
        for di, d in enumerate(doses):
            for r in range(CFG["DOSE_R"]):
                arr[di, r] = predict_logits(m, core.deg_dose(Vs, d, np.random.default_rng(50_000 + 100 * di + r)), amp=False)
        out[key] = arr
    np.savez_compressed(fp, **out)
print("dose-scaling finished")

dose-scaling finished


In [ ]:
%%writefile analysis.py
# ===== Analysis: reads saved predictions, writes tables (CSV) and figures (PNG, 300 dpi) =====
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import expit
import core

TYPE_LABEL = {"dose": "Dose reduction", "scatter": "Scatter", "contrast": "Subject contrast\n(kVp-like)",
              "overexp": "Over-exposure\n+ saturation", "motion": "Motion blur"}
SCORES = ["msp", "tta_std", "combo", "noise"]
SCORE_LABEL = {"msp": "Max-softmax (1-conf)", "tta_std": "TTA instability", "combo": "MSP + TTA",
               "noise": "Image noise (model-free)"}

def _load(ROOT, CFG):
    y = np.load(os.path.join(ROOT, "preds", "y_test.npy"))
    temps = json.load(open(os.path.join(ROOT, "tables", "temperatures.json")))
    conds = core.conditions()
    data = {}
    for (c, l, p) in conds:
        data[(c, l)] = dict(np.load(os.path.join(ROOT, "preds", f"cond_{c}_L{l}.npz")))
    return y, temps, conds, data

def metrics_tables(ROOT, CFG, y, temps, conds, data):
    rows = []
    for (c, l, p) in conds:
        d = data[(c, l)]
        for key, T in temps.items():
            name, seed = key.rsplit("_s", 1)
            for calib, TT in [("raw", 1.0), ("ts", T)]:
                m = core.metrics_from_logits(d[f"{key}__logit"], y, T=TT, conf_thr=CFG["CONF_THR"])
                rows.append(dict(model=name, seed=int(seed), cond=c, level=l, param=p, calib=calib, **m))
    long = pd.DataFrame(rows)
    long.to_csv(os.path.join(ROOT, "tables", "metrics_long.csv"), index=False)
    mets = ["acc", "bal_acc", "auroc", "ece", "brier", "nll", "conf_err_rate", "frac_err_confident", "mean_conf_wrong", "sens", "spec"]
    summ = long.groupby(["model", "calib", "cond", "level", "param"], dropna=False)[mets].agg(["mean", "std"])
    summ.columns = [f"{a}_{b}" for a, b in summ.columns]
    summ = summ.reset_index()
    summ.to_csv(os.path.join(ROOT, "tables", "metrics_summary.csv"), index=False)
    # compact paper table: clean vs worst level of each type (TS-calibrated)
    sel = summ[(summ.calib == "ts") & ((summ.level == 0) | (summ.level == 5))].copy()
    fmt = lambda r, k: f"{r[k+'_mean']:.3f} ± {0 if np.isnan(r[k+'_std']) else r[k+'_std']:.3f}"
    tab = pd.DataFrame([{"model": r.model, "condition": "clean" if r.level == 0 else f"{r.cond} L5 ({r.param})",
                         "AUROC": fmt(r, "auroc"), "Acc": fmt(r, "acc"), "ECE": fmt(r, "ece"),
                         "Confident-error rate": fmt(r, "conf_err_rate"),
                         "Errors that are confident": fmt(r, "frac_err_confident")} for _, r in sel.iterrows()])
    tab.to_csv(os.path.join(ROOT, "tables", "table_clean_vs_worst.csv"), index=False)
    return long, summ

def score_matrices(key, T, conds, data, y, conf_thr):
    """Returns dict of (N, C) score matrices and conf_err (N, C) for one model run."""
    S = {s: [] for s in SCORES}; CE = []; ER = []
    for (c, l, p) in conds:
        d = data[(c, l)]
        pr = expit(d[f"{key}__logit"] / T)
        pred = (pr >= 0.5).astype(int); conf = np.where(pred == 1, pr, 1 - pr)
        msp = 1 - conf; tstd = d[f"{key}__tta_std"]
        S["msp"].append(msp); S["tta_std"].append(tstd); S["combo"].append(msp + tstd); S["noise"].append(d["noise"])
        CE.append((pred != y) & (conf >= conf_thr)); ER.append(pred != y)
    return {s: np.stack(v, 1) for s, v in S.items()}, {"conf_err": np.stack(CE, 1), "error": np.stack(ER, 1)}

LOSS_LABEL = {"conf_err": "confident error", "error": "any error"}

def gate_analysis(ROOT, CFG, y, temps, conds, data):
    trial_rows, curve_rows, aurc_rows, accept_rows = [], [], [], []
    alphas = {"conf_err": CFG["ALPHAS_CONF"], "error": CFG["ALPHAS_ERR"]}
    for key, T in temps.items():
        name, seed = key.rsplit("_s", 1)
        S, LOSS = score_matrices(key, T, conds, data, y, CFG["CONF_THR"])
        for loss_name, L in LOSS.items():
            for s in SCORES:
                sc, ce = S[s].ravel(), L.ravel()
                order = np.argsort(sc, kind="stable")
                cum = np.cumsum(ce[order]) / np.arange(1, len(ce) + 1)
                cov = np.arange(1, len(ce) + 1) / len(ce)
                pick = np.unique(np.linspace(0, len(ce) - 1, 101).astype(int))
                for i in pick:
                    curve_rows.append(dict(model=name, seed=int(seed), loss=loss_name, score=s, coverage=cov[i], selective_risk=cum[i]))
                aurc_rows.append(dict(model=name, seed=int(seed), loss=loss_name, score=s, aurc=float(cum.mean())))
                for a in alphas[loss_name]:
                    res = core.run_crc(S[s], L, a, n_trials=CFG["N_TRIALS"], seed=int(seed))
                    for r in res:
                        trial_rows.append(dict(model=name, seed=int(seed), loss=loss_name, score=s, alpha=a, **r))
                    lam = float(np.median([r["lambda"] for r in res]))
                    acc = (S[s] <= lam).mean(axis=0)
                    for ci, (c, l, p) in enumerate(conds):
                        accept_rows.append(dict(model=name, seed=int(seed), loss=loss_name, score=s, alpha=a, cond=c, level=l, accept_rate=acc[ci]))
    trials = pd.DataFrame(trial_rows).rename(columns={"cond_conf_err_among_accepted": "risk_among_accepted"})
    trials.to_csv(os.path.join(ROOT, "tables", "crc_trials.csv"), index=False)
    summ = trials.groupby(["model", "loss", "score", "alpha"]).agg(
        mean_test_risk=("test_risk", "mean"), std_test_risk=("test_risk", "std"),
        mean_accept=("accept_rate", "mean"), std_accept=("accept_rate", "std"),
        risk_no_gate=("risk_no_gate", "mean"),
        risk_among_accepted=("risk_among_accepted", "mean")).reset_index()
    summ["frac_trials_risk_gt_alpha"] = [
        float((trials[(trials.model == r.model) & (trials.loss == r.loss) & (trials.score == r.score) & (trials.alpha == r.alpha)].test_risk > r.alpha).mean())
        for r in summ.itertuples()]
    summ.to_csv(os.path.join(ROOT, "tables", "crc_summary.csv"), index=False)
    pd.DataFrame(curve_rows).to_csv(os.path.join(ROOT, "tables", "risk_coverage_curves.csv"), index=False)
    aurc = pd.DataFrame(aurc_rows); aurc.to_csv(os.path.join(ROOT, "tables", "aurc.csv"), index=False)
    aurc.groupby(["model", "loss", "score"]).aurc.agg(["mean", "std"]).reset_index().to_csv(
        os.path.join(ROOT, "tables", "aurc_summary.csv"), index=False)
    acc = pd.DataFrame(accept_rows); acc.to_csv(os.path.join(ROOT, "tables", "gate_accept_by_condition.csv"), index=False)
    return trials, summ, acc

def dose_analysis(ROOT, CFG):
    fp = os.path.join(ROOT, "preds", "dose_scaling.npz")
    if not os.path.exists(fp): return None
    z = dict(np.load(fp)); doses = z.pop("doses"); z.pop("sub")
    rows = []
    for key, arr in z.items():                       # arr: (D, R, N)
        name, seed = key.rsplit("_s", 1)
        var = arr.var(axis=1, ddof=1)                # (D, N)
        mv, medv = var.mean(axis=1), np.median(var, axis=1)
        for di, d in enumerate(doses):
            rows.append(dict(model=name, seed=int(seed), dose=d, mean_logit_var=mv[di], median_logit_var=medv[di]))
    df = pd.DataFrame(rows); df.to_csv(os.path.join(ROOT, "tables", "dose_scaling.csv"), index=False)
    fits = []
    for (m, s), g in df.groupby(["model", "seed"]):
        for regime, mask in [("all doses", g.dose > 0), ("high dose d >= 1", g.dose >= 1), ("low dose d <= 1", g.dose <= 1)]:
            gg = g[mask]
            slope, icpt = np.polyfit(np.log(gg.dose), np.log(gg.median_logit_var), 1)
            fits.append(dict(model=m, seed=s, regime=regime, slope=slope, intercept=icpt))
    fits = pd.DataFrame(fits); fits.to_csv(os.path.join(ROOT, "tables", "dose_scaling_fits.csv"), index=False)
    return df, fits

# ---------------- figures ----------------
def _save(fig, ROOT, name):
    fig.savefig(os.path.join(ROOT, "figures", name), dpi=300, bbox_inches="tight"); plt.close(fig)

def figures(ROOT, CFG, y, temps, conds, data, summ, trials, acc, dose):
    # Fig 1: degradation examples
    ex = {(c, l): data[(c, l)]["examples"].astype(np.float32) for (c, l, p) in conds}
    types = list(core.DEGRADATIONS)
    for ei, lab in enumerate(["normal", "pneumonia"]):
        fig, ax = plt.subplots(len(types), 6, figsize=(12, 2.1 * len(types)))
        for r, t in enumerate(types):
            for l in range(6):
                img = ex[("clean", 0)][ei] if l == 0 else ex[(t, l)][ei]
                ax[r, l].imshow(img, cmap="gray", vmin=0, vmax=1); ax[r, l].axis("off")
                prm = None if l == 0 else core.DEGRADATIONS[t][0][l - 1]
                ax[r, l].set_title("clean" if l == 0 else f"L{l} ({prm})", fontsize=8)
            ax[r, 0].text(-0.15, 0.5, TYPE_LABEL[t], transform=ax[r, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
        _save(fig, ROOT, f"fig1_degradation_examples_{lab}.png")

    # Fig 2-4: metric vs severity per degradation type
    for met, lab in [("auroc", "AUROC"), ("ece", "ECE"), ("conf_err_rate", "Confident-error rate"),
                     ("frac_err_confident", "Fraction of errors made confidently")]:
        for calib in ["ts", "raw"]:
            fig, ax = plt.subplots(1, len(types), figsize=(3.2 * len(types), 3), sharey=True)
            for i, t in enumerate(types):
                for mdl in CFG["MODELS"]:
                    g = summ[(summ.model == mdl) & (summ.calib == calib) & ((summ.cond == t) | (summ.cond == "clean"))].sort_values("level")
                    mu, sd = g[f"{met}_mean"].values, np.nan_to_num(g[f"{met}_std"].values)
                    ax[i].plot(g.level, mu, marker="o", ms=3, label=mdl); ax[i].fill_between(g.level, mu - sd, mu + sd, alpha=0.2)
                ax[i].set_title(TYPE_LABEL[t], fontsize=9); ax[i].set_xlabel("Severity level"); ax[i].set_xticks(range(6))
            ax[0].set_ylabel(lab); ax[-1].legend(fontsize=7)
            _save(fig, ROOT, f"fig2_{met}_vs_severity_{calib}.png")

    # Fig 3: reliability diagrams (first model, seed 0)
    key = list(temps)[0]; T = temps[key]
    sel = [("clean", 0)] + [(t, 5) for t in types]
    fig, ax = plt.subplots(1, len(sel), figsize=(3 * len(sel), 3), sharey=True)
    for i, (c, l) in enumerate(sel):
        pr = expit(data[(c, l)][f"{key}__logit"] / T); pred = (pr >= 0.5).astype(int)
        conf = np.where(pred == 1, pr, 1 - pr); corr = (pred == y)
        bins = np.linspace(0.5, 1, 11); idx = np.clip(np.digitize(conf, bins[1:-1], right=True), 0, 9)
        accs = [corr[idx == b].mean() if (idx == b).any() else np.nan for b in range(10)]
        ax[i].bar((bins[:-1] + bins[1:]) / 2, accs, width=0.045, edgecolor="k")
        ax[i].plot([0.5, 1], [0.5, 1], "r--", lw=1); ax[i].set_xlim(0.5, 1); ax[i].set_ylim(0, 1)
        ax[i].set_title("clean" if c == "clean" else f"{TYPE_LABEL[c]} L5".replace("\n", " "), fontsize=8); ax[i].set_xlabel("Confidence")
    ax[0].set_ylabel("Accuracy"); fig.suptitle(f"Reliability diagrams ({key}, temperature-scaled)", fontsize=10)
    _save(fig, ROOT, "fig3_reliability_diagrams.png")

    # Fig 4: dose scaling
    if dose is not None:
        df, fits = dose
        fig, ax = plt.subplots(figsize=(4.5, 3.5))
        for mdl in CFG["MODELS"]:
            g = df[df.model == mdl].groupby("dose").median_logit_var.agg(["mean", "std"]).reset_index()
            ax.errorbar(g.dose, g["mean"], yerr=np.nan_to_num(g["std"]), marker="o", ms=4, capsize=2, label=mdl)
        d = np.array(sorted(df.dose.unique())); dmax = d.max(); ref = df[df.dose == dmax].median_logit_var.mean()
        ax.plot(d, ref * dmax / d, "k--", lw=1, label="slope −1 (delta method)")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Dose fraction d"); ax.set_ylabel("Median Var[logit]")
        ax.legend(fontsize=7); _save(fig, ROOT, "fig4_dose_variance_scaling.png")

    curves = pd.read_csv(os.path.join(ROOT, "tables", "risk_coverage_curves.csv"))
    alphas = {"conf_err": CFG["ALPHAS_CONF"], "error": CFG["ALPHAS_ERR"]}
    for loss_name in ["conf_err", "error"]:
        # Fig 5: risk-coverage curves
        fig, ax = plt.subplots(1, len(CFG["MODELS"]), figsize=(3.6 * len(CFG["MODELS"]), 3.2), sharey=True)
        ax = np.atleast_1d(ax)
        for i, mdl in enumerate(CFG["MODELS"]):
            for s in SCORES:
                g = curves[(curves.model == mdl) & (curves.loss == loss_name) & (curves.score == s)].groupby("coverage").selective_risk.mean().reset_index()
                ax[i].plot(g.coverage, g.selective_risk, label=SCORE_LABEL[s])
            ax[i].set_title(mdl, fontsize=9); ax[i].set_xlabel("Coverage (fraction accepted)")
        ax[0].set_ylabel(f"Rate of {LOSS_LABEL[loss_name]}s among accepted"); ax[-1].legend(fontsize=7)
        _save(fig, ROOT, f"fig5_risk_coverage_{loss_name}.png")
        # Fig 6: CRC guarantee check
        fig, ax = plt.subplots(1, len(alphas[loss_name]), figsize=(3.8 * len(alphas[loss_name]), 3.2))
        ax = np.atleast_1d(ax)
        for i, a in enumerate(alphas[loss_name]):
            g = trials[(trials.alpha == a) & (trials.loss == loss_name)]
            vals = [g[g.score == s].test_risk.values for s in SCORES]
            ax[i].boxplot(vals, showfliers=False); ax[i].set_xticks(range(1, len(SCORES) + 1))
            ax[i].set_xticklabels([SCORE_LABEL[s] for s in SCORES], rotation=30, ha="right", fontsize=7)
            ax[i].axhline(a, color="r", ls="--", lw=1); ax[i].set_title(f"α = {a}", fontsize=9)
        ax[0].set_ylabel(f"Realised {LOSS_LABEL[loss_name]} risk"); _save(fig, ROOT, f"fig6_crc_risk_control_{loss_name}.png")
        # Fig 7: gate acceptance vs severity (middle alpha)
        a = alphas[loss_name][len(alphas[loss_name]) // 2]
        fig, ax = plt.subplots(1, len(types), figsize=(3.2 * len(types), 3), sharey=True)
        for i, t in enumerate(types):
            for s in ["msp", "tta_std", "combo"]:
                g = acc[(acc.loss == loss_name) & (acc.alpha == a) & (acc.score == s) & ((acc.cond == t) | (acc.cond == "clean"))].groupby("level").accept_rate.mean()
                ax[i].plot(g.index, g.values, marker="o", ms=3, label=SCORE_LABEL[s])
            ax[i].set_title(TYPE_LABEL[t], fontsize=9); ax[i].set_xlabel("Severity level"); ax[i].set_xticks(range(6))
        ax[0].set_ylabel(f"Acceptance rate (α={a})"); ax[-1].legend(fontsize=7)
        _save(fig, ROOT, f"fig7_gate_acceptance_by_severity_{loss_name}.png")

def run_all(ROOT, CFG):
    y, temps, conds, data = _load(ROOT, CFG)
    long, summ = metrics_tables(ROOT, CFG, y, temps, conds, data); print("metrics tables written")
    trials, crc_summ, acc = gate_analysis(ROOT, CFG, y, temps, conds, data); print("conformal gate analysis written")
    dose = dose_analysis(ROOT, CFG); print("dose scaling written")
    figures(ROOT, CFG, y, temps, conds, data, summ, trials, acc, dose); print("figures written")
    pd.set_option("display.width", 200)
    print("\n== Clean vs worst severity (TS-calibrated) ==")
    print(pd.read_csv(os.path.join(ROOT, "tables", "table_clean_vs_worst.csv")).to_string(index=False))
    print("\n== Conformal gate summary ==")
    print(crc_summ.round(4).to_string(index=False))
    if dose is not None:
        print("\n== Dose scaling slopes (delta method predicts -1) ==")
        print(dose[1].groupby(["model", "regime"]).slope.agg(["mean", "std"]).round(3))


Overwriting analysis.py


In [ ]:
# ===== FIX: find conditions with NaN/Inf predictions and recompute them in full precision (fp32) =====
bad = []
for ci, (cname, lvl, prm) in enumerate(CONDS):
    fp = os.path.join(ROOT, "preds", f"cond_{cname}_L{lvl}.npz")
    z = np.load(fp)
    nonfinite = {k: int((~np.isfinite(z[k].astype(np.float64))).sum())
                 for k in z.files if (~np.isfinite(z[k].astype(np.float64))).any()}
    if nonfinite:
        print(f"{cname} L{lvl}:", nonfinite)
        bad.append((ci, cname, lvl, prm))
print("Conditions with NaN/Inf:", len(bad))

for ci, cname, lvl, prm in bad:
    t0 = time.time()
    Vd = core.apply_condition(V_test, cname, prm, seed=1000 + ci)   # identical degraded images (same seed)
    save = dict(noise=core.noise_estimate(Vd), examples=Vd[ex_idx].astype(np.float16))
    for key, m in MODELS.items():
        T = TEMPS[key]
        base = predict_logits(m, Vd, amp=False)
        tta = np.stack([predict_logits(m, Vd, tta_seed=10_000 * ci + k, amp=False) for k in range(CFG["K_TTA"])])
        p_all = expit(np.concatenate([base[None], tta]) / T)
        save[f"{key}__logit"] = base
        save[f"{key}__tta_std"] = p_all.std(axis=0)
        save[f"{key}__tta_flip"] = ((p_all[1:] >= 0.5) != (p_all[0] >= 0.5)).mean(axis=0)
    left = sum(int((~np.isfinite(v.astype(np.float64))).sum()) for v in save.values())
    np.savez_compressed(os.path.join(ROOT, "preds", f"cond_{cname}_L{lvl}.npz"), **save)
    print(f"recomputed {cname} L{lvl} in fp32 ({time.time()-t0:.0f}s), remaining NaN/Inf: {left}")

with open(os.path.join(ROOT, "logs", "fp32_recomputed.json"), "w") as f:
    json.dump([dict(cond=c, level=l) for _, c, l, _ in bad], f, indent=2)

dose L4: {'efficientnet_b0_s1__logit': 54, 'efficientnet_b0_s1__tta_std': 54}
dose L5: {'efficientnet_b0_s1__logit': 9, 'efficientnet_b0_s1__tta_std': 10}
Conditions with NaN/Inf: 2
recomputed dose L4 in fp32 (56s), remaining NaN/Inf: 0
recomputed dose L5 in fp32 (56s), remaining NaN/Inf: 0


In [ ]:
# ===== 8. Analysis: tables, conformal gate, figures =====
import importlib, analysis
importlib.reload(analysis)
analysis.run_all(ROOT, CFG)

metrics tables written
conformal gate analysis written
dose scaling written
figures written

== Clean vs worst severity (TS-calibrated) ==
             model          condition         AUROC           Acc           ECE Confident-error rate Errors that are confident
   efficientnet_b0              clean 0.988 ± 0.005 0.931 ± 0.019 0.053 ± 0.020        0.041 ± 0.017             0.582 ± 0.081
   efficientnet_b0 contrast L5 (0.25) 0.812 ± 0.040 0.715 ± 0.066 0.223 ± 0.095        0.187 ± 0.085             0.630 ± 0.165
   efficientnet_b0     dose L5 (0.02) 0.544 ± 0.067 0.628 ± 0.006 0.340 ± 0.060        0.319 ± 0.097             0.855 ± 0.251
   efficientnet_b0   motion L5 (19.0) 0.774 ± 0.110 0.652 ± 0.032 0.264 ± 0.135        0.220 ± 0.158             0.605 ± 0.413
   efficientnet_b0   overexp L5 (8.0) 0.827 ± 0.044 0.723 ± 0.042 0.205 ± 0.041        0.168 ± 0.048             0.607 ± 0.150
   efficientnet_b0   scatter L5 (4.0) 0.445 ± 0.035 0.590 ± 0.030 0.172 ± 0.079        0.105 ± 0.10

In [ ]:
# ===== 9. Package results for upload to the chat =====
import shutil, platform, sklearn, torchvision
env = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
           sklearn=sklearn.__version__, numpy=np.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(env, open(os.path.join(ROOT, "logs", "environment.json"), "w"), indent=2)
pkg = "/content/IJEEEMI_CXR_results"
if os.path.exists(pkg): shutil.rmtree(pkg)
shutil.copytree(ROOT, pkg, ignore=shutil.ignore_patterns("*.pt"))
zip_path = shutil.make_archive(os.path.join(ROOT, "IJEEEMI_CXR_results"), "zip", pkg)
print("Upload this file to the chat:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")

Upload this file to the chat: /content/drive/MyDrive/IJEEEMI_CXR/IJEEEMI_CXR_results.zip (15.5 MB)


In [ ]:
# ===== 10. Print compact results to paste into the chat =====
import pandas as pd
T = lambda f: pd.read_csv(os.path.join(ROOT, "tables", f))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 500)
print("### ENV"); print(json.load(open(os.path.join(ROOT, "logs", "environment.json"))))
print("### TRAINING"); print(T("training_summary.csv").round(4).to_string(index=False))
s = T("metrics_summary.csv")
cols = ["model","cond","level","param","auroc_mean","auroc_std","acc_mean","bal_acc_mean","ece_mean","ece_std",
        "conf_err_rate_mean","conf_err_rate_std","frac_err_confident_mean","sens_mean","spec_mean"]
print("### METRICS (temperature-scaled)"); print(s[s.calib=="ts"][cols].round(4).to_string(index=False))
print("### METRICS (raw, clean + L5)"); r = s[(s.calib=="raw") & (s.level.isin([0,5]))]; print(r[cols].round(4).to_string(index=False))
print("### CRC GATE"); print(T("crc_summary.csv").round(4).to_string(index=False))
print("### AURC"); print(T("aurc_summary.csv").round(5).to_string(index=False))
a = T("gate_accept_by_condition.csv"); a = a[a.alpha.isin([0.01, 0.10])]
print("### ACCEPT RATE BY CONDITION (mean over models; columns = severity level)")
print(a.groupby(["loss","score","alpha","cond","level"]).accept_rate.mean().unstack("level").round(3).to_string())
print("### DOSE SCALING"); d = T("dose_scaling.csv")
print(d.groupby(["model","dose"])[["mean_logit_var","median_logit_var"]].mean().to_string())
print(T("dose_scaling_fits.csv").groupby(["model","regime"]).slope.agg(["mean","std"]).round(3).to_string())

### ENV
{'python': '3.13.16', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'sklearn': '1.6.1', 'numpy': '2.1.3', 'gpu': 'Tesla T4'}
### TRAINING
                  key  best_val_auroc  best_epoch  train_minutes  n_params  temperature
mobilenet_v3_large_s0          0.9993          10         2.4674   4203313       1.8685
mobilenet_v3_large_s1          0.9994          14         2.1860   4203313       2.2149
mobilenet_v3_large_s2          0.9992           7         1.9057   4203313       1.5297
   efficientnet_b0_s0          0.9997           4         2.2926   4008829       1.0356
   efficientnet_b0_s1          0.9998           2         1.7830   4008829       0.6156
   efficientnet_b0_s2          0.9996           2         1.7892   4008829       0.6603
          resnet18_s0          0.9997           9         2.0728  11177025       1.3567
          resnet18_s1          0.9996           8         1.8993  11177025       1.1677
          resnet18_s2          0.9996          11   